In [ ]:
# TO DO also: 
# data loader for final number of partitions: force it to be what specified instead of always half??;

# (consider using run_worker_sweep for simplicity??)
# DEBUG NEW KMEANS (check matrices stuff)?  ---> try with "old" kmeans??

# clean up code (image saving, etc)?
# re-run final, clean code?

# K-means|| distribuito su Dask — Analisi

Notebook di orchestrazione: avvia il cluster, carica il dataset, esegue i test (singola run o benchmark su più combinazioni) e salva i risultati in `./results`.

Moduli usati:
- `kmeans_parallel.py` — algoritmo k-means|| distribuito
- `launch_cluster.py` — avvio/spegnimento del cluster Dask via SSH
- `data_loader.py` — caricamento del dataset
- `benchmark.py` — esecuzione dei test e salvataggio risultati

#### Useful: to kill existing Dask processes (from bash):
```bash
for ip in 10.67.22.194 10.67.22.254 10.67.22.34 10.67.22.145 10.67.22.121 10.67.22.192 10.67.22.18 10.67.22.187 10.67.22.48; do
    echo "Killing on $ip..."
    ssh -o StrictHostKeyChecking=no ubuntu@"$ip" "pkill -9 -f dask-scheduler; pkill -9 -f dask-worker; pkill -9 -f dask_ssh" 
done
```

## 1. Import

In [ ]:
import numpy as np
import pandas as pd
from src.kmeans_parallel import kmeans_parallel
from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset
from src.benchmark import run_single_test, run_benchmark, calculate_inertia, combinations_fn

import time

## 2. Parametri configurabili

Modifica qui i valori per cambiare numero di worker, `k`, `l` (oversampling factor) e `r` (numero di round dell'inizializzazione parallela).

In [ ]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers

In [ ]:
# --- Algoritmo k-means|| ---
#K = 500                # numero di cluster finali
#L = 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
#R = 10                  # numero di round dell'inizializzazione parallela
#MAX_ITER_FIT = 100      # iterazioni massime della fase di Lloyd's (fit)

SEED = 42

In [ ]:
# --- Dataset ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL="https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH_10PC   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH_10PC = '/tmp/kddcup_data_shards' # directory of shard Parquet files on master

RAW_GZ_PATH_FULL = "/home/ubuntu/backup/libero_development/data/kddcup_data_full.gz"
PARQUET_PATH_FULL = '/tmp/kddcup_data_full_shards'

# column names of KDD dataset, from source code of the above sklearn function;
# "protocol_type","service","flag" are non-numeric so they will be dropped later,
# as will be "label" and the constant column 'num_outbound_cmds' 
# (10% dataset might have more constant columns such as 'is_host_login')
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

## 3. Start cluster

In [ ]:
# DO NOT RUN if already existing!
cluster, client = launch_cluster(N_WORKERS)

#### Connect to cluster IF INSTEAD already exsiting:

In [ ]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

In [ ]:
client.scheduler_info()

## 4. Load dataset

In [ ]:
#10 percent:

start=time.time()
X_bag_10_percent, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   parquet_path_workers=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=False)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

In [ ]:
# if want to go back to original coordinates later:
#print(mean_ar, '\n\n\n') 
#print(std_ar)

In [ ]:
# full:

start=time.time()
X_bag_full, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_FULL,#DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_FULL,#RAW_GZ_PATH,
                   parquet_path=PARQUET_PATH_FULL,#PARQUET_PATH,
                   parquet_path_workers=PARQUET_PATH_FULL,#PARQUET_PATH,
                   col_names=COL_NAMES,
                   force_download=True)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

#### Check that loaded correctly

In [ ]:
# this was done earlier, for full dataset:

In [ ]:
# global variable used in data_loader.py 
print(f"Number of initial features (should be 42): {len(COL_NAMES)}")

In [ ]:
n_features_blocks = X_bag_full.shape[1]
print("Number of features per block:", n_features_blocks)
# Should be 37 for the full dataset (see below)

In [ ]:
n_points = int(X_bag_full.shape[0])
n_features = X_bag_full.shape[1]
print(f"Points: {n_points}, Features: {n_features}\n(Full dataset should contain 4898431 samples with 37 (=42 -1 label -3 non numeric -1 constant feature) numerical, non-constant features)")

In [ ]:
has_nan = bool(np.isnan(X_bag_full).any().compute())
has_inf = bool(np.isinf(X_bag_full).any().compute())
print(f"Has NaN: {has_nan}, Has Inf: {has_inf}\n(Both must be False)")

In [ ]:
total = X_bag_full.sum(axis=0).compute()
mean = total / n_points
means_vector=np.any(np.abs(mean)>1e-3)
print(f"Any column mean significantly different from zero: {means_vector==True}\n(Must be False)")

In [ ]:
sq_total = (X_bag_full ** 2).sum(axis=0).compute()

std = np.sqrt(sq_total / n_points - mean**2)
std_minus_one_vector=np.abs(std - 1)
print(f"Any column where std significantly different from one: {np.any(std_minus_one_vector > 1e-3)}\n(Must be False)")

In [ ]:
sample_rows = X_bag_full.partitions[0].compute()[:3]
for i, row in enumerate(sample_rows):
    print(f"Row {i}: shape={row.shape}, first 5 values={row[:5]}")

In [ ]:
# additional sanity check
print("Type of X_bag_full:", type(X_bag_full))
print("Number of partitions:", X_bag_full.npartitions)

In [ ]:
total_bytes = X_bag_full.nbytes
print(f"Total dataset size: {total_bytes / 1e6:.2f} MB")

exp_size=n_features * n_points * 8
print(f"Expected size: {exp_size / 1e6} MB")

## 5. Single run

Runs a single combination of parameters (the ones defined in section 2) and prints the cost and execution time.

In [ ]:
K = 129 #500                # numero di cluster finali
L = 4*K # 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
R = 5                 # numero di round dell'inizializzazione parallela
MAX_ITER_FIT = 10      # iterazioni massime della fase di Lloyd's (fit)

dataset_bag=X_bag_10_percent
# consider other values?? eg in original paper, or K=23 for real life dataset

In [ ]:
# consider other values?? eg in original paper, or K=23 for real life dataset

start=time.time()
result, _ = run_single_test(
    client,
    k=K, l=L, r=R,
    num_partitions=NUM_PARTITIONS,#*2,
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
    X_bag=dataset_bag,
    track_convergence=True,
    track_centroids=True
)
result
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

In [ ]:
# 29 s WITHOUT tracking anything

In [ ]:
#cost =567232.43 
#cost_ten=cost * 1e-10
#print(f"(Cost equivalent to {cost_ten :.2f} * 10^10)")
minutes=elapsed/60
print(f"(Time elapsed equivalent to {minutes:.2f} minutes)")

In [ ]:
# if tracked cost:

import matplotlib.pyplot as plt

iterations = range(1, len(result["cost_history"]) + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iterations, result["cost_history"], marker="o", markersize=4, linewidth=2, color="tab:blue")
ax.set_xlabel("Iteration")
ax.set_ylabel("Cost (inertia)")
ax.set_title(f"K-means convergence (k={result['k']})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
centroids_table = pd.DataFrame({
    "round": range(1, len(result["n_centroids_history"]) + 1),
    "n_centroids": result["n_centroids_history"],
})
centroids_table.style.set_caption(f"l = {result['l']}")

Compare with:
* Cost $19 \cdot 10^{10}$ ($k=500$, $l=0.5 k$, but $r=5$ instead - table 3.).
  HOWEVER, note that here we standardized data beforehand!
* Running time $69.0 \text{ min}$ for the same values (table 4)

## 6. Multi-combination benchmark

Testa più combinazioni di `(n_workers, partitions, l_over_k, r)` per diversi valori di `k`, e salva tutto in `./results`. 

**Nota:** il numero di worker effettivamente attivi nel cluster è quello impostato con `launch_cluster` in sezione 3 — la colonna `workers` qui sotto serve solo per etichettare/loggare i risultati, non riavvia il cluster.

### Varying number of workers

#### On 10% dataset

In [ ]:
# write as function? but this is clearer and not that long

In [ ]:
# as best combination (for cost) in figure 5.1 in original paper:

R = 5 

l_over_k = 4

K_VALUES=[129]

MAX_ITER_FIT=10 # unimportant to get perfect convergence,
                # as we're investigating only ideal number of workers (wrt to computation time)
                # ok, but must be greater than 10. However, with 10% it's quick. Mattia

In [ ]:
n_workers=[2,4,6,8]

avg_iters=3

In [ ]:
shutdown_cluster(cluster,client)

In [ ]:
frames = []
for n in n_workers:
    cluster, client = launch_cluster(n)
    X_bag_10_percent, _ = load_dataset(n_partitions=8*n,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   parquet_path_workers=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=False)  # rebuild bag on new client - perhaps unnenecessary??
    
    df_n = run_benchmark(client, X_bag=X_bag_10_percent,
                         combinations=combinations_fn(n_workers=n,l_over_k=l_over_k,r=R),
                         # function assigning appropriate values (in particular n partitions = 8 * n workers)
                         k_values=K_VALUES, label=f"{n}_workers_10_percent", 
                         max_iter_fit=MAX_ITER_FIT, seed=SEED,
                         averaging_iterations=avg_iters)
    
    df_n["num_workers"] = n
    frames.append(df_n)
    shutdown_cluster(cluster, client)

df_all = pd.concat(frames, ignore_index=True)

In [ ]:
import os

In [ ]:
RESULTS_DIR = "/home/ubuntu/Project/libero_development/results"
FIGURES_DIR= "/home/ubuntu/Project/libero_development/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
csv_path = os.path.join(RESULTS_DIR, f"worker_sweep_{time.strftime('%Y%m%d%H%M%S')}_10pc.csv")
figure_path=os.path.join(FIGURES_DIR, f"worker_sweep_{time.strftime('%Y%m%d%H%M%S')}_10pc.png")
df_all.to_csv(csv_path, index=False)
print(f"Saved to: {csv_path}")

In [ ]:
# Group by number of workers, compute mean and std of time
stats = df_all.groupby("num_workers")["time"].agg(["mean", "std"]).reset_index()

plt.figure(figsize=(8, 5))
plt.errorbar(
    stats["num_workers"], 
    stats["mean"], 
    yerr=stats["std"], 
    marker="o", 
    capsize=5,
    linestyle="-",
    label="Mean ± std"
)
plt.xlabel("Number of Workers")
plt.ylabel("Time (seconds)")
plt.title("Computation time vs Number of active workers\n(10 LLoyd iterations, 10% dataset)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(figure_path, dpi=150)
plt.show()

In [ ]:
# and, can check directly to see that similar final cost for any number of workers

####On full dataset

In [ ]:
import importlib
import src.kmeans_parallel
importlib.reload(src.kmeans_parallel)
from src.kmeans_parallel import kmeans_parallel

In [ ]:
n_workers=[5,4,3,2]

avg_iters=1

In [ ]:
# other params same as above

In [ ]:
shutdown_cluster(cluster,client)

In [ ]:
import dask

In [ ]:
frames = []
for n in n_workers:
    #dask.config.set({"distributed.scheduler.work-stealing": True})
    cluster, client = launch_cluster(n)
    mt=client.run(lambda: __import__('ctypes').CDLL("libc.so.6").malloc_trim(0))
    print(mt)
    X_bag_full, _ = load_dataset(n_partitions=8*n,
                   client=client,
                   dataset_url=DATASET_URL_FULL,
                   raw_gz_path=RAW_GZ_PATH_FULL,
                   parquet_path=PARQUET_PATH_FULL,
                   parquet_path_workers=PARQUET_PATH_FULL,
                   col_names=COL_NAMES,
                   force_download=False)  # rebuild bag on new client
    
    df_n = run_benchmark(client, X_bag=X_bag_full,
                         combinations=combinations_fn(n_workers=n,l_over_k=l_over_k,r=R),
                         # function assigning appropriate values (in particular n partitions = 8 * n workers)
                         k_values=K_VALUES, label=f"{n}_workers_full", 
                         max_iter_fit=MAX_ITER_FIT, seed=SEED,
                         averaging_iterations=avg_iters)
    
    df_n["num_workers"] = n
    frames.append(df_n)
    shutdown_cluster(cluster, client)

df_all = pd.concat(frames, ignore_index=True)

In [ ]:
# after the crash, check what happened right before death
client.get_worker_logs()

In [ ]:
client.upload_file('/home/ubuntu/Project/libero_development/src/kmeans_parallel.py')

In [ ]:
def check_version():
    import src.kmeans_parallel
    import inspect
    return inspect.getsource(src.kmeans_parallel._update_state)

print(client.run(check_version))

In [ ]:
# save results
csv_path = os.path.join(RESULTS_DIR, f"worker_sweep_{time.strftime('%Y%m%d%H%M%S')}_full.csv")
figure_path=os.path.join(FIGURES_DIR, f"worker_sweep_{time.strftime('%Y%m%d%H%M%S')}_full.png")
df_all.to_csv(csv_path, index=False)
print(f"Saved to: {csv_path}")

In [ ]:
# plot result

# Group by number of workers, compute mean and std of time
stats = df_all.groupby("num_workers")["time"].agg(["mean", "std"]).reset_index()

plt.figure(figsize=(8, 5))
plt.errorbar(
    stats["num_workers"], 
    stats["mean"], 
    yerr=stats["std"], 
    marker="o", 
    capsize=5,
    linestyle="-",
    label="Mean ± std"
)
plt.xlabel("Number of Workers")
plt.ylabel("Time (seconds)")
plt.title("Computation time vs Number of active workers\n(10 LLoyd iterations, full dataset)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(figure_path, dpi=150)
plt.show()

### Varying number of partitions

#### On 10% dataset

In [ ]:
dataset_bag= X_bag_10_percent
avg_iters=10

In [ ]:
# other params same as above

In [ ]:
combos = [                      
    (N_WORKERS, 32, l_over_k, R),   # under-partitioned
    (N_WORKERS, 64, l_over_k, R),   # balanced (1 part/thread)
    #(N_WORKERS, 65, l_over_k, R),   # imbalanced
    (N_WORKERS, 128, l_over_k, R),  # over-partitioned
]

In [ ]:
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="num_partitions_10_percent",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters)

In [ ]:
df

#### On full dataset

In [ ]:
# IMPORTANT NOTE: for sure comparison, at this point, we shut down and recreate cluster:

In [ ]:
shutdown_cluster(cluster,client)

In [ ]:
cluster, client= launch_cluster(N_WORKERS)

In [ ]:
# as best combination (for cost) in figure 5.1 in original paper:

dataset_bag= X_bag_full
avg_iters=3 # for faster results (small stdev anyways)

In [ ]:
# other params same as above

In [ ]:
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="num_partitions_full",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters)

### Varying l_over_k 

#### On full dataset - (Tables 3 and 4 in original paper)

In [ ]:
dataset_bag= X_bag_full

R=5

num_threads= 64 # as found in previous code # NUM_PARTITIONS*2

K_VALUES=[500,1000]

MAX_ITER_FIT=100 # for best convergence (conv criterion??)

avg_iters=10

In [ ]:
combos = [
    (N_WORKERS, num_threads, 0.5, R)
    (N_WORKERS, num_threads, 1, R),  #FATTO
    (N_WORKERS, num_threads, 2, R),
    (N_WORKERS, num_threads, 10, R),
]

In [ ]:
df_results = run_benchmark(
    client, X_bag=dataset_bag,
    combinations=combos,
    k_values=K_VALUES,
    label="l_over_k_full",
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
    averaging_iterations = avg_iters
)

In [ ]:
df_results

#### 6.1 Visualization of results

Per vedere i risultati ottenuti precedentemente

In [ ]:
from src.benchmark_analysis import BenchmarkAnalyzer

analyzer = BenchmarkAnalyzer(
    data_path="/home/ubuntu/Project/libero_development/notebooks/results/kddcup99_benchmark20260825193532.csv",
    output_dir="figures",
    facet_cols=["k"],              # una figura per ogni valore (combinazione) di queste colonne
    x_col="partitions",               # variabile sull'asse x
    metrics=["time"],       # colonne di cui calcolare mean/std e plottare
)
grouped = analyzer.compute_grouped_stats(groupby_cols=["k", "partitions"])
analyzer.print_summary(grouped)
analyzer.plot_all(grouped)

#### On 10% dataset - (Figure 5.1 in original paper)

## 7. Comparison to standard k-means algorithm

## Cluster shutdown

Da eseguire a fine lavoro, o prima di rilanciare `launch_cluster` con un `N_WORKERS` diverso.

In [ ]:
shutdown_cluster(cluster, client)